In [ ]:
import dill as pickle
model_dir = "/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out"
with open(f"{model_dir}/semantic_cat_classifier.pkl", "rb") as f:
    model = pickle.load(f)

In [ ]:
import fasttext
import fasttext.util


ft = fasttext.load_model(f'{model_dir}/cc.en.300.bin')


# now add semantic categories to patient transcripts
import string
for pt, df in patient_transcripts.items():
    if "FinalClusterID" in df.columns:
        continue
    df["CollapsedWord"] = None
    df["CleanedWord"] = None
    print("starting", pt)
    w2v_embeddings = []
    for idx, row in df.iterrows():
        try:
            # get collapsed word
            word = ''
            for field in row.index:
                if field.lower().startswith('speaker') and isinstance(row[field], str):
                    word = row[field]
            df.loc[idx, 'CollapsedWord'] = word
            word_format = word.rstrip(string.punctuation).lower()
            df.loc[idx, 'CleanedWord'] = word_format
            embedding = ft.get_word_vector(word_format)
            # try to output embedding, otherwise just nan vector
        except Exception as e:
            print(e)
            embedding = np.full_like(embedding, np.nan)
        w2v_embeddings.append(embedding)
    w2v_embeddings = np.vstack(w2v_embeddings)

    threshold = 0.8
    data_probs = model.predict_proba(w2v_embeddings)
    top_idx = data_probs.argmax(axis=1)
    conf = data_probs[np.arange(data_probs.shape[0]), top_idx]
    keep = conf >= threshold
    top_idx[~keep] = -1
    df["FinalClusterID"] = top_idx
    patient_transcripts[pt] = df
